# Lesson 2: Mapping the City from OpenStreetMap

**Planning question: What exists on the ground?**

Before you plan anything new, you need to know what is already there.
OpenStreetMap (OSM) is a free map of the world, built by volunteers. It holds buildings, roads, parks, schools, shops and much more.

In this notebook you will download these layers for **Indiranagar** in Bengaluru and combine them into one base map.

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q osmnx

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

## Settings

Instead of the whole city, which would be millions of objects, we take a circle around one point.
`CENTER` holds the latitude and longitude of Indiranagar. `DIST` is the distance from the centre in metres.

To find coordinates for another place, right-click it in Google Maps and click the numbers to copy them.

In [ ]:
CENTER = (12.9719, 77.6412)   # (latitude, longitude) of Indiranagar
DIST = 1500                   # metres
UTM_CRS = "EPSG:32643"

## Step 1: Buildings

OSM stores information as **tags**: a key and a value, such as `building=house` or `leisure=park`.
Setting `{"building": True}` asks for every object that has a building tag, whatever its value.

In [ ]:
buildings = ox.features_from_point(CENTER, tags={"building": True}, dist=DIST)
buildings = buildings[buildings.geom_type.isin(["Polygon", "MultiPolygon"])]
print("Number of buildings:", len(buildings))

The `building` column tells us what kind of building it is. Many are just tagged `yes`,
because the mapper did not record the type. This is common in OSM.

In [ ]:
buildings["building"].value_counts().head(10)

## Step 2: Streets

`graph_from_point` downloads the street network. We convert it to a table of lines (`edges`) so we can map it.

In [ ]:
G = ox.graph_from_point(CENTER, dist=DIST, network_type="all")
streets = ox.graph_to_gdfs(G, nodes=False)
print("Number of street segments:", len(streets))
streets["highway"].astype(str).value_counts().head(8)

## Step 3: Parks and amenities

Now we download parks, plus a few amenities that matter for daily life.

In [ ]:
parks = ox.features_from_point(CENTER, tags={"leisure": "park"}, dist=DIST)
parks = parks[parks.geom_type.isin(["Polygon", "MultiPolygon"])]

amenity_types = ["school", "hospital", "clinic", "restaurant", "bank", "place_of_worship"]
amenities = ox.features_from_point(CENTER, tags={"amenity": amenity_types}, dist=DIST)

print("Parks:", len(parks))
amenities["amenity"].value_counts()

Some amenities are mapped as points and others as building outlines. To show them all the same way,
we turn every one into a single point at its centre (its **centroid**). We do this in UTM so the centres are accurate.

In [ ]:
amenity_points = amenities[["amenity", "name", "geometry"]].to_crs(UTM_CRS)
amenity_points["geometry"] = amenity_points.centroid
amenity_points = amenity_points.to_crs("EPSG:4326")

## Step 4: A layered base map

GIS maps are built up in **layers**, drawn from the bottom to the top. This is the figure-ground style map that architects know well.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
parks.plot(ax=ax, color="#9bd39b")
streets.plot(ax=ax, color="#bbbbbb", linewidth=0.6)
buildings.plot(ax=ax, color="#333333")
amenity_points.plot(ax=ax, column="amenity", markersize=25, legend=True,
                    legend_kwds={"loc": "lower left"})
ax.set_title("Indiranagar: buildings, streets, parks and amenities")
ax.set_axis_off()
plt.show()

## Step 5: Park area per person (a quick check)

How much park space is there? We add up the park areas in square metres.
Indian planning guidelines often suggest around 10 to 12 square metres of open space per person.
The population figure below is a rough example only. Replace it with census data for real work.

In [ ]:
park_area_m2 = parks.to_crs(UTM_CRS).area.sum()
study_area_m2 = 3.1416 * DIST ** 2
example_population = 80_000

print(f"Park area: {park_area_m2 / 10_000:,.1f} hectares")
print(f"Share of study area that is park: {park_area_m2 / study_area_m2 * 100:.1f}%")
print(f"Park area per person: {park_area_m2 / example_population:.1f} square metres")

## Step 6: Save the layers

In [ ]:
buildings[["building", "geometry"]].to_file("indiranagar_osm.gpkg", layer="buildings", driver="GPKG")
streets[["highway", "length", "geometry"]].astype({"highway": str}).to_file("indiranagar_osm.gpkg", layer="streets", driver="GPKG")
parks[["name", "geometry"]].to_file("indiranagar_osm.gpkg", layer="parks", driver="GPKG")
amenity_points.to_file("indiranagar_osm.gpkg", layer="amenities", driver="GPKG")
print("Saved indiranagar_osm.gpkg")

## What this means for planning

- OSM is a fast, free way to build a base map for site analysis.
- Its quality varies. Always check it against a site visit or official data.
  Missing tags, such as many buildings tagged only as `yes`, are a common gap.
- Even a simple count of parks and amenities can show where a neighbourhood lacks services.

## Try it yourself

1. Change `CENTER` to Jayanagar `(12.9250, 77.5938)` and compare its building and park counts with Indiranagar.
2. Add `"bus_station"` or `"library"` to `amenity_types`.
3. Download water bodies with `tags={"natural": "water"}` and add them to the layered map.